# SNN Brain Playground
Same training framework as other playgrounds; only the model plugin differs.


In [1]:
from __future__ import annotations

from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "train_headless.py").exists() and (REPO_ROOT.parent / "train_headless.py").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import math
import jax
import jax.numpy as jnp

from brains.config import load_runtime_spec
from brains.jax_trainer import apply_runtime_spec
from brains.models import (
    NotebookModel,
    apply_notebook_model,
    register_inline_policy_factory,
    register_notebook_model,
)
from brains.models.notebook_framework import train_and_save_with_progress


In [2]:
class SimpleSNNPolicy:
    """Stateful spiking policy: dense input + recurrent + readout, with reset-on-spike."""

    def __init__(self, input_size: int, output_size: int):
        self.input_size = input_size
        self.output_size = output_size
        self.hidden = 24
        self.decay = jnp.float32(0.9)
        self.threshold = jnp.float32(0.15)

    def init_params(self, key):
        k1, k2, k3, k4 = jax.random.split(key, 4)
        return {
            "w_in": jax.random.normal(k1, (self.hidden, self.input_size), dtype=jnp.float32) * jnp.float32(1.0 / math.sqrt(self.input_size)),
            "w_rec": jax.random.normal(k2, (self.hidden, self.hidden), dtype=jnp.float32) * jnp.float32(1.0 / math.sqrt(self.hidden)),
            "w_out": jax.random.normal(k3, (self.output_size, self.hidden), dtype=jnp.float32) * jnp.float32(1.0 / math.sqrt(self.hidden)),
            "b": jax.random.uniform(k4, (self.hidden,), dtype=jnp.float32, minval=-0.05, maxval=0.05),
            "b_out": jnp.zeros((self.output_size,), dtype=jnp.float32),
        }

    def zero_state(self):
        return jnp.zeros((self.hidden,), dtype=jnp.float32)

    def step(self, params, state, obs, key, noise_scale):
        current = jnp.matmul(params["w_in"], obs) + jnp.matmul(params["w_rec"], state) + params["b"]
        v = (self.decay * state) + current
        spikes = (v > self.threshold).astype(jnp.float32)
        next_state = jnp.where(spikes > 0.0, jnp.float32(0.0), v)
        logits = jnp.matmul(params["w_out"], spikes) + params["b_out"]
        key, noise_key = jax.random.split(key)
        noisy = logits + (jax.random.normal(noise_key, (self.output_size,), dtype=jnp.float32) * jnp.maximum(noise_scale, 0.0))
        return next_state, jnp.tanh(noisy), key


def build_snn_policy(spec, model_definition):
    del spec
    return SimpleSNNPolicy(model_definition.input_size, model_definition.output_size)


SNN_ENTRYPOINT = register_inline_policy_factory("notebook_simple_snn_v1", build_snn_policy)
SNN_ENTRYPOINT


'inline:notebook_simple_snn_v1'

In [3]:
MODEL_TYPE = "notebook_simple_snn_v1"
LOG_ID = datetime.now(timezone.utc).strftime("simple_snn_%Y%m%dT%H%M%SZ")
TRAIN_GENERATIONS = 2
SEED = 11

base_spec = load_runtime_spec(REPO_ROOT / "configs/default.yaml")
model = NotebookModel(
    model_type=MODEL_TYPE,
    architecture="simple_snn",
    description="Stateful spiking policy defined inline in this notebook.",
    input_size=48,
    output_size=4,
    parameter_count=None,
    policy_entrypoint=SNN_ENTRYPOINT,
    control_mode="motor_targets",
)
register_notebook_model(model, spec=base_spec, persist=False)

spec = apply_notebook_model(base_spec, model)
spec = replace(spec, name="simple-snn-default")
apply_runtime_spec(spec)

{"model": spec.model.type, "architecture": spec.model.architecture, "control_mode": spec.control.mode}


W0501 11:10:36.871259 2651382 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


{'model': 'notebook_simple_snn_v1',
 'architecture': 'simple_snn',
 'control_mode': 'motor_targets'}

In [4]:
result = train_and_save_with_progress(
    spec=spec,
    repo_root=REPO_ROOT,
    model_type=MODEL_TYPE,
    log_id=LOG_ID,
    seed=SEED,
    generations=TRAIN_GENERATIONS,
    print_step_progress=False,
)
result



generation 1/2 start


generation 1/2 done | mean_reward=-108.2740 | best_reward=-0.9335


generation 2/2 start


generation 2/2 done | mean_reward=-124.9473 | best_reward=1.5577


{'model_id': 'notebook_simple_snn_v1_simple_snn_20260501T151036Z',
 'log_id': 'simple_snn_20260501T151036Z',
 'latest': '/Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim/checkpoints/notebook_simple_snn_v1_simple_snn_20260501T151036Z/latest.npz',
 'generation': 2,
 'mean_reward': -124.94729614257812,
 'best_reward': 1.5577316284179688,
 'elapsed_s': 125.50135129199771,
 'visible_artifacts': ['notebook_simple_snn_v1_simple_snn_20260501T151036Z',
  'notebook_command_brain_v1_command_brain_20260501T151039Z',
  'notebook_shared_trunk_v1_shared_trunk_20260501T130232Z']}